# Notebook 4 — Entrenamiento del modelo final para producción



## 0. Setup — imports y rutas

In [1]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

DATA_PATH    = Path('../data/btc_1d_data_2018_to_2025.csv')
RESULTS_DIR  = Path('../results')
APP_DIR      = Path('../app')
MODEL_PATH   = APP_DIR / 'model.joblib'

WINDOW_SIZE      = 30
ANNUALIZATION    = 365
N_STEPS_FORECAST = 7

pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

## 1. Leer la combinación ganadora del ranking

`ranking_arch.csv` tiene una fila por `(n_lag, arch)`. El ganador global es el de menor `test_rmse_avg`. Sus `hidden_layers` y el valor de `n_lag` se usan como `n_steps_input` del modelo final. También leemos `config.joblib` para reutilizar los mismos `MLP_PARAMS` que se usaron en la CV, garantizando coherencia entre el modelo de validación y el de producción.

In [2]:
ranking = pd.read_csv(RESULTS_DIR / 'ranking_arch.csv')
config  = joblib.load(RESULTS_DIR / 'config.joblib')

winner = ranking.sort_values('test_rmse_avg').iloc[0]
N_STEPS_INPUT  = int(winner['n_lag'])
HIDDEN_LAYERS  = tuple(eval(winner['hidden_layers']))  # '[64]' -> (64,)
MLP_PARAMS     = dict(config['mlp_params'])

print(f'Combinacion ganadora:')
print(f'  n_lag          = {N_STEPS_INPUT}')
print(f'  arch           = {winner["arch"]}')
print(f'  hidden_layers  = {HIDDEN_LAYERS}')
print(f'  test_rmse_avg  = {winner["test_rmse_avg"]:.4f}')
print(f'  test_mae_avg   = {winner["test_mae_avg"]:.4f}')
print(f'  n_folds_won    = {int(winner["n_folds_won"])} / 5')
print(f'\nMLP_PARAMS (heredados de nb2):')
for k, v in MLP_PARAMS.items():
    print(f'  {k:22s} = {v}')

Combinacion ganadora:
  n_lag          = 7
  arch           = h[32,32,128]_lr1e-03
  hidden_layers  = (32, 32, 128)
  test_rmse_avg  = 0.0746
  test_mae_avg   = 0.0493
  n_folds_won    = 0 / 5

MLP_PARAMS (heredados de nb2):
  solver                 = adam
  activation             = relu
  max_iter               = 500
  early_stopping         = True
  validation_fraction    = 0.1
  n_iter_no_change       = 20
  tol                    = 0.0001
  learning_rate_init     = 0.001


**Comentario — lectura de la combinación ganadora**

Confirmación del ganador global del `ranking_arch.csv` del grid expandido:

- **`n_lag = 7`**, **`hidden_layers = (32, 32, 128)`**, `test_rmse_avg = 0.0746`, `test_mae_avg = 0.0493`.
- **`n_folds_won = 0/5`** — la arquitectura **NUNCA ganó un fold individual** en la CV de nb2, pero tiene el mejor `test_rmse_avg` promediando sobre las 15 mediciones (3 seeds × 5 folds). Esa es la propiedad deseada para producción: **consistencia, no brillo puntual**. Una arch que gana el Fold 1 con RMSE 0.05 pero explota en el Fold 5 con RMSE 0.12 puede tener promedio peor que esta, que entrega ~0.075 en todos los folds.

**`MLP_PARAMS` heredados vía `config['mlp_params']`**: `lr=1e-3` (el único valor barrido en nb2), `max_iter=500`, `early_stopping=True`, `n_iter_no_change=20`. 

## 2. Cargar la serie y recomputar la volatilidad

Mismos pasos que el notebook 2 para garantizar reproducibilidad: carga de `Date` y `Close`, limpieza temporal, cálculo de `LogReturn` y `Volatility` rolling anualizada por √365.

In [3]:
btc = pd.read_csv(DATA_PATH)
assert {'Date', 'Close'}.issubset(btc.columns)
btc = btc[['Date', 'Close']].copy()
btc['Date']  = pd.to_datetime(btc['Date'], errors='coerce')
btc['Close'] = pd.to_numeric(btc['Close'], errors='coerce').astype(float)
btc = (btc
       .dropna(subset=['Date', 'Close'])
       .sort_values('Date')
       .drop_duplicates(subset='Date', keep='first')
       .reset_index(drop=True))
btc['LogReturn']  = np.log(btc['Close'] / btc['Close'].shift(1))
btc['Volatility'] = (btc['LogReturn']
                     .rolling(window=WINDOW_SIZE)
                     .std() * np.sqrt(ANNUALIZATION))
btc = btc.dropna(subset=['LogReturn', 'Volatility']).reset_index(drop=True)

series = btc['Volatility'].values.astype(float)
print(f'Longitud de la serie de volatilidad: {len(series):,}')
print(f'Rango de fechas: {btc["Date"].min().date()} -> {btc["Date"].max().date()}')

Longitud de la serie de volatilidad: 2,891
Rango de fechas: 2018-02-01 -> 2025-12-31


**Comentario — serie recomputada**

**2 891 filas**, rango `2018-02-01 → 2025-12-31`.

## 3. Construir todas las ventanas (X, y) de la serie completa

Para cada índice `i` válido: `X[i] = series[i : i+n_in]`, `y[i] = series[i+n_in : i+n_in+n_out]`. En producción **no** se hace split — queremos usar toda la información disponible para el modelo final. La validación de generalización ya quedó cubierta por la CV del notebook 2.

In [4]:
def build_windows(series, n_in, n_out):
    N = len(series) - n_in - n_out + 1
    X = np.empty((N, n_in), dtype=float)
    y = np.empty((N, n_out), dtype=float)
    for i in range(N):
        X[i] = series[i : i + n_in]
        y[i] = series[i + n_in : i + n_in + n_out]
    return X, y

X, y = build_windows(series, N_STEPS_INPUT, N_STEPS_FORECAST)
print(f'X.shape = {X.shape}   (muestras, n_steps_input)')
print(f'y.shape = {y.shape}   (muestras, n_steps_forecast)')

X.shape = (2878, 7)   (muestras, n_steps_input)
y.shape = (2878, 7)   (muestras, n_steps_forecast)


**Comentario — ventanas de entrenamiento**

- `X.shape = (2878, 7)` — 2 878 ventanas de input de 7 días.
- `y.shape = (2878, 7)` — 2 878 vectores target de 7 pasos.
- Fórmula: N = 2 891 − 7 (input) − 7 (output) + 1 = 2 878. Cuadra.

**Volumen de training relativo a la CV**: cada fold de nb2 tenía ~500-600 muestras de train. El modelo final entrena con 2 878 — **5-6× más datos**. En teoría esto debería darnos un modelo más preciso que el fold ganador de la CV (más datos = menos varianza en los pesos estimados).


## 4. Escalado y entrenamiento final

`StandardScaler` ajustado sobre **todas** las ventanas (para producción, a diferencia de la CV). `MLPRegressor` con los hiperparámetros heredados del notebook 2 y `random_state=0` fijo. El early stopping interno (`validation_fraction=0.1`) sigue activo para que el modelo no sobreajuste aunque entrenemos con más datos.

In [5]:
scaler_x = StandardScaler().fit(X)
scaler_y = StandardScaler().fit(y)

X_s = scaler_x.transform(X)
y_s = scaler_y.transform(y)

with warnings.catch_warnings():
    warnings.simplefilter('ignore', ConvergenceWarning)
    model = MLPRegressor(
        hidden_layer_sizes=HIDDEN_LAYERS,
        random_state=0,
        **MLP_PARAMS,
    )
    model.fit(X_s, y_s)

print(f'Entrenamiento final completado.')
print(f'  n_iter    = {model.n_iter_}')
print(f'  converged = {model.n_iter_ < MLP_PARAMS["max_iter"]}')
print(f'  n_layers (incl. output) = {model.n_layers_}')

Entrenamiento final completado.
  n_iter    = 43
  converged = True
  n_layers (incl. output) = 5


**Comentario — entrenamiento final**

- **`n_iter = 43`** — el modelo convergió en 43 épocas, muy por debajo del `max_iter=500`. Señal de buena elección de `learning_rate_init` (1e-3 es el sweet spot para esta arq): gradientes que progresan rápido sin oscilar.
- **`converged = True`** — el early_stopping cortó por falta de mejora en el val interno (20 épocas consecutivas sin mejora en R²), no por topar iters. Es decir, el modelo se saturó genuinamente.
- **`n_layers_ = 5`** — 1 input + 3 hidden (32, 32, 128) + 1 output = 5 capas totales. Arquitectura replicada fielmente desde el ranking.

**Por qué convergió tan rápido** (43 vs los ~100-200 que vimos en los folds de nb2): tenemos 5× más datos de train en el modelo final, así que los gradientes son más estables y menos ruidosos → convergencia más rápida. Es una ventaja de entrenar con toda la serie.



## 5. Métrica de referencia sobre los datos de entrenamiento

No es una medida de generalización (ya la tuvimos en la CV), pero sirve como sanity check: el modelo debe ajustar razonablemente bien. Reportamos RMSE/MAE por horizonte sobre toda la serie.

In [6]:
yhat_s = model.predict(X_s)
yhat   = scaler_y.inverse_transform(yhat_s)

mae_h  = np.mean(np.abs(y - yhat), axis=0)
rmse_h = np.sqrt(np.mean((y - yhat) ** 2, axis=0))
mape_h = np.mean(np.abs((y - yhat) / (np.abs(y) + 1e-8)), axis=0) * 100

ref = pd.DataFrame({
    'horizon':  [f'h_{h+1}' for h in range(N_STEPS_FORECAST)],
    'MAE':      mae_h,
    'RMSE':     rmse_h,
    'MAPE (%)': mape_h,
})
print('Sanity check sobre los datos de entrenamiento (NO es test de generalizacion):')
print(ref.to_string(index=False))
print(f'\nPromedios -> MAE={mae_h.mean():.4f}  RMSE={rmse_h.mean():.4f}  MAPE={mape_h.mean():.2f}%')

Sanity check sobre los datos de entrenamiento (NO es test de generalizacion):
horizon      MAE     RMSE  MAPE (%)
    h_1 0.023495 0.046223  4.186923
    h_2 0.033818 0.064400  5.970669
    h_3 0.042879 0.078030  7.642733
    h_4 0.050555 0.089606  8.989173
    h_5 0.058535 0.101134 10.408754
    h_6 0.065511 0.111260 11.581186
    h_7 0.071463 0.120398 12.521924

Promedios -> MAE=0.0495  RMSE=0.0873  MAPE=8.76%


**Comentario — métricas de referencia en training (¡NO es generalización!)**

 **test de generalización**.  ¿qué es?

Es el error de predicción del modelo sobre TODAS las 2 878 ventanas (el 90 % que entrenó + el 10 % de val interna del early_stopping). No es error de generalización porque la mayoría de estas muestras el modelo las vio; no es training error puro tampoco porque incluye la val interna que el modelo usó para cortar.



| métrica | nb4 "train" (2 878 ventanas) | nb2 CV test (lag=7) |
|---------|------------------------------|---------------------|
| RMSE    | 0.0873                       | 0.0745              |
| MAE     | 0.0495                       | 0.0494              |
| MAPE    | 8.76 %                       | 9.07 %              |

El MAE es prácticamente idéntico y el MAPE es mejor en nb4. Pero **el RMSE de nb4 es PEOR** (0.087 vs 0.075). Contraintuitivo.

**Explicación**: las 2 878 ventanas incluyen los períodos de régimen extremo (COVID-2020, Luna-2022, FTX-2022) con vol > 1.5. El MLP final los ve y los modela, pero su predicción en esos picos falla fuerte (residuos de 0.3+) y el RMSE cuadra-doble ese error. La CV de nb2, al fragmentar la serie en 5 folds, **separa** esos picos en folds individuales cuyo promedio atenúa el outlier. Por eso el RMSE promediado sobre 5 folds da una imagen más optimista que el RMSE agregado sobre la serie completa.


## 6. Serialización del artefacto canónico

Dict con las cinco keys que `app/api.py::_unpack_artifact` reconoce. La API carga este archivo en el arranque, expone `expected_lags = n_steps_input` y responde 7 predicciones por petición al endpoint `/predict`.

In [7]:
APP_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    'model':            model,
    'scaler_x':         scaler_x,
    'scaler_y':         scaler_y,
    'n_steps_input':    N_STEPS_INPUT,
    'n_steps_forecast': N_STEPS_FORECAST,
}
joblib.dump(artifact, MODEL_PATH)
print(f'Guardado: {MODEL_PATH.resolve()}')
print(f'Tamano del artefacto: {MODEL_PATH.stat().st_size / 1024:.1f} KB')

Guardado: C:\Users\juana\btc-volatility-mlops\app\model.joblib
Tamano del artefacto: 158.1 KB


## 7. Sanity check del artefacto serializado

Cargamos el modelo desde disco (como lo hará la API), emulamos el unpacking de `api.py::_unpack_artifact` y ejercitamos el pipeline de inferencia con una ventana sintética. Esto debe:

- Cargar sin errores.
- Devolver una predicción con shape `(1, 7)`.
- Producir valores finitos (ni NaN ni inf).

In [8]:
loaded = joblib.load(MODEL_PATH)
_model    = loaded['model']
_scaler_x = loaded['scaler_x']
_scaler_y = loaded['scaler_y']
_n_in     = loaded['n_steps_input']
_n_out    = loaded['n_steps_forecast']

# Emulamos una peticion: ventana sintetica con n_in valores positivos
X_test   = np.linspace(0.3, 0.8, _n_in).reshape(1, -1)
X_test_s = _scaler_x.transform(X_test)
y_test   = _model.predict(X_test_s)
if y_test.ndim == 1:
    y_test = y_test.reshape(1, -1)
y_test   = _scaler_y.inverse_transform(y_test)

assert y_test.shape == (1, 7), f'Shape incorrecto: {y_test.shape}'
assert np.all(np.isfinite(y_test)), 'Hay predicciones no finitas'

print(f'Entrada sintetica (volatilidades): {X_test[0].round(3).tolist()}')
print(f'Prediccion h=1..7              : {y_test[0].round(4).tolist()}')
print(f'\nArtefacto sano y consumible por la API.')

Entrada sintetica (volatilidades): [0.3, 0.383, 0.467, 0.55, 0.633, 0.717, 0.8]
Prediccion h=1..7              : [0.8139, 0.8093, 0.8229, 0.8435, 0.8595, 0.8988, 0.8741]

Artefacto sano y consumible por la API.


**Comentario — sanity check del artefacto serializado**

Emulación del consumo del modelo por `app/api.py`:

- **Entrada sintética**: volatilidades crecientes linealmente de 0.3 a 0.8 (pasando de "calma" a "stress moderado").
- **Predicción h=1..7**: `[0.81, 0.81, 0.82, 0.84, 0.86, 0.90, 0.87]`.

**Lectura**: el modelo predice que la volatilidad **persiste y se amplifica** tras una entrada que ya venía subiendo. Esto es consistente con el **clustering de volatilidad** observado en nb1 (ACF positiva de retornos²): cuando la vol está en ascenso, el próximo día tiene más probabilidad de vol alta que de vol baja.

Los valores predichos (0.81-0.90) están en el rango realista de la distribución histórica (media 0.60, p75 0.71, max 2.12). **Ninguna predicción explota** (no vemos 5.0 o −1.0), confirmando que los scalers se aplicaron correctamente en ambos sentidos (forward y inverse).

**Validaciones que pasaron**:
- `shape == (1, 7)` ✓
- `all(np.isfinite(y))` ✓
- Carga del joblib sin errores ✓
- Forward pass sin warnings ✓


## 8. Final

- Modelo final listo en `../app/model.joblib` (combinación `lag` + `arch` elegida automáticamente del ranking).
- La API FastAPI (`app/api.py`) y los tests (`tests/test_model.py`, `tests/test_api.py`) 
**Para validar end-to-end desde la raíz del repo:**

```bash
pytest tests/ -v
uvicorn app.api:app --reload --port 8000
```

El health-check `GET /` debe reportar `expected_lags = N_STEPS_INPUT` y `n_steps_forecast = 7`.